# Dawn

## Loading the Observation Log

In [1]:
import sqlite3
import pandas as pd

In [2]:
connection = sqlite3.connect("file:../data/observations.db?mode=ro", uri=True)

## Run Counts by Source and Status

In [3]:
run_counts = pd.read_sql_query(
    """
    SELECT source, status, COUNT(*) AS runs
    FROM source_runs
    GROUP BY source, status
    ORDER BY source, status
    """,
    connection,
)

In [4]:
print("Shape :", run_counts.shape)

Shape : (5, 3)


In [5]:
run_counts

,source,status,runs
0,himalayas,ok,1
1,jobicy,ok,1
2,remoteok,ok,1
3,remotive,ok,1
4,weworkremotely,ok,1


In [6]:
run_counts.groupby("status")["runs"].sum()

status
ok    5
Name: runs, dtype: int64

In [7]:
print("Total runs :", run_counts["runs"].sum())

Total runs : 5


## Runs per Source

In [8]:
runs_per_source = run_counts.groupby("source")["runs"].sum()
runs_per_source

source
himalayas         1
jobicy            1
remoteok          1
remotive          1
weworkremotely    1
Name: runs, dtype: int64

In [9]:
print("Sources :", len(runs_per_source))

Sources : 5


In [10]:
print("Most runs for one source :", runs_per_source.max())

Most runs for one source : 1


In [11]:
print("Sources with at least two runs :", (runs_per_source >= 2).sum())

Sources with at least two runs : 0


## Observed Rows by Source

In [12]:
row_counts = pd.read_sql_query(
    """
    SELECT source, COUNT(*) AS observed_rows
    FROM observations
    GROUP BY source
    ORDER BY source
    """,
    connection,
)

In [13]:
print("Shape :", row_counts.shape)

Shape : (5, 2)


In [14]:
row_counts

,source,observed_rows
0,himalayas,20
1,jobicy,100
2,remoteok,99
3,remotive,11
4,weworkremotely,100


In [15]:
print("Total observed rows :", row_counts["observed_rows"].sum())

Total observed rows : 330


In [16]:
connection.close()

## Wrap-Up

The observation log has 5 runs and all 5 have the status ok. Each of the 5 sources has 1 run, so 0 sources have two runs to compare. The log holds 330 observed rows (himalayas 20, jobicy 100, remoteok 99, remotive 11, weworkremotely 100). These are rows stored by the capture, not a count of what each source returned.

One run per source is not enough for survival curves. A posting has to be seen in one run and then looked for again in a later run before anything can be said about how long it stayed. So this notebook fits no model and gives no lifetime estimates. The counts are from the local database on 2026-11-13 and will change once more captures are saved.